# HW2: Who should the bank call?

**Name:** _your name_ &nbsp;&nbsp; **Student ID:** _your id_

See `README.md` for the scenario, requirements and grading. This starter loads the data, sets up MLflow and writes the submission files; the rest is yours. Put your final model in `hw2_submission.py`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # keep OpenMP threads modest on shared machines

import json

import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

import hw2_submission as sub

SEED = 0
EXPERIMENT = "hw2-call-list"
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # browse: uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_experiment(EXPERIMENT)

/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/homeworks/hw2-call-list/mlruns/1', creation_time=1790703362663, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790703362663, lifecycle_stage='active', name='hw2-call-list', tags={}, trace_location=None, workspace='default'>

In [2]:
bank = fetch_openml(data_id=1461, as_frame=True)
NAMES = ["age", "job", "marital", "education", "default", "balance", "housing", "loan", "contact",
         "day", "month", "duration", "campaign", "pdays", "previous", "poutcome"]
X = bank.data.copy()
X.columns = NAMES                                  # OpenML stores them as V1..V16
y = (bank.target == "2").astype(int).to_numpy()    # 1 = subscribed to a term deposit
for c in X.select_dtypes(exclude="number").columns:
    X[c] = X[c].astype("category")                 # exactly what the grader passes to your pipeline
print(X.shape, f"positive rate {y.mean():.3f}  (rows are in chronological order)")
X.head()

(45211, 16) positive rate 0.117  (rows are in chronological order)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown


## 1. Data and protocol

In [3]:
# TODO

## 2. Candidates, baseline and equal-budget tuning

In [4]:
# TODO

## 3. Final evaluation with uncertainty

Helper (provided): a paired bootstrap interval for the difference between two models on the **same** test set, as in Week 5 ("Are these differences real?"). Use it as-is or write your own.


In [5]:
import numpy as np
from sklearn.metrics import roc_auc_score


def paired_bootstrap_ci(y_true, score_a, score_b, metric=roc_auc_score, n_boot=2000, seed=0, level=0.95):
    """Bootstrap interval for metric(A) - metric(B) on the same test set.

    Resamples test rows with replacement and recomputes both metrics on each
    resample, so the pairing between the two models is kept.
    Returns (observed difference, lower bound, upper bound).
    """
    y_true, score_a, score_b = map(np.asarray, (y_true, score_a, score_b))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:      # a resample needs both classes
            continue
        diffs.append(metric(y_true[idx], score_a[idx]) - metric(y_true[idx], score_b[idx]))
    alpha = (1 - level) / 2
    lo, hi = np.quantile(diffs, [alpha, 1 - alpha])
    return metric(y_true, score_a) - metric(y_true, score_b), lo, hi


# example: diff, lo, hi = paired_bootstrap_ci(y_test, proba_best, proba_baseline)


In [6]:
# TODO

## 4. Will it hold up on the next campaign?

In [7]:
# TODO

## 5. Check the submission module

In [8]:
try:
    model = sub.build_pipeline()
    print("module OK:", sub.CHOSEN_MODEL, type(model).__name__, "on", len(sub.FEATURES), "features")
except NotImplementedError as e:
    print("not done yet:", e)

not done yet: TODO: build your pipeline


## 6. Answers and export

A reported number is `{"value": float, "run_id": "<MLflow run id>", "metric": "<metric name in that run>"}`.
Family names: `"logistic_regression"`, `"random_forest"`, `"hist_gradient_boosting"`, `"lightgbm"`, `"xgboost"`, `"mlp"`, `"other"`.

In [9]:
answers = {
    "student_id": None,                    # str
    "leak_column": None,                   # str: the column you excluded as a leak
    "positive_rate_last_20pct": None,      # float: subscription rate in the last 20% of rows
    "forward_auc_last_20pct": None,        # reported number: your chosen pipeline trained on the first 80% of rows,
                                           #   ROC AUC on the last 20%
    "models_compared": None,               # list of family names
    "configs_per_model": None,             # int: hyperparameter configurations evaluated per family
    "chosen_model": None,                  # family name
    "test_auc": None,                      # reported number: chosen model, your held-out test set
    "baseline_test_auc": None,             # reported number: logistic-regression baseline, same test set
    "chosen_minus_baseline_auc_ci": None,  # [low, high]: 95% interval for the AUC difference
    "expected_auc_new_clients": None,      # float: AUC you expect on new clients like those in the data
    "memo": {
        "decision": None,                  # "deploy_model" | "deploy_baseline" | "do_not_deploy"
        "expected_lift_top10": None,       # float: subscription rate in the top 10% by score / overall rate
        "main_risk": None,                 # "campaign_conditions_change" | "leakage" | "class_imbalance" |
                                           # "data_quality" | "none"
        "justification": "",               # at most 150 words for the call-centre manager (not graded)
    },
}
with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)
mlflow.search_runs(experiment_names=[EXPERIMENT]).to_csv("mlflow_runs.csv", index=False)
print("wrote answers.json and mlflow_runs.csv")

wrote answers.json and mlflow_runs.csv


## AI-use statement

TODO: which AI tools you used, for what, and how you checked their output ("None" is valid, but write a full sentence).